# 🧠 CNN Model Training & Hyperparameter Tuning Notebook
This notebook documents the training workflow, data augmentation experiments, CNN architecture construction, and evaluation metrics for the **Smart Waste Classification** project.

In [ ]:
import sys
import os
sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

from src.data_preprocessing import load_and_preprocess_data, create_data_generator
from src.model import build_cnn_model, build_mobilenet_model
from src.utils import save_class_indices

%matplotlib inline
print("TensorFlow Version:", tf.__version__)

## 1. Load Preprocessed Data via OpenCV Pipeline

In [ ]:
data_dir = "../data"
img_size = (224, 224)

# Load preprocessed images
X, y, class_names = load_and_preprocess_data(data_dir=data_dir, img_size=img_size, max_samples_per_class=200)
print(f"Loaded {len(X)} images across classes: {class_names}")

# Split into Train (80%) and Validation (20%)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=np.argmax(y, axis=1))
print(f"Train samples: {len(X_train)} | Validation samples: {len(X_val)}")

## 2. Build & Inspect CNN Architecture

In [ ]:
model = build_cnn_model(input_shape=(224, 224, 3), num_classes=len(class_names), learning_rate=0.001)
model.summary()

## 3. Data Augmentation & Model Training

In [ ]:
batch_size = 32
epochs = 10

train_gen = create_data_generator(X_train, y_train, batch_size=batch_size)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6)
]

history = model.fit(
    train_gen,
    steps_per_epoch=len(X_train) // batch_size,
    epochs=epochs,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

## 4. Evaluation & Confusion Matrix

In [ ]:
# Predictions on validation set
y_val_pred_probs = model.predict(X_val)
y_val_pred = np.argmax(y_val_pred_probs, axis=1)
y_val_true = np.argmax(y_val, axis=1)

# Metrics Report
print("Classification Report:")
print(classification_report(y_val_true, y_val_pred, target_names=class_names))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_val_true, y_val_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()